# Know Your Data: Exploring and Preparing Image Datasets at the Wildwood Farm Park

**Objective:** run the setup, meet the park, and see the three things you will be able to do at the end.

<table width="100%"><tr><td bgcolor="#0B6E79"><font color="#FFFFFF">
<font size="5"><b>Know Your Data: Exploring and Preparing Image Datasets
at the Wildwood Farm Park</b></font><br><br>
<b>At the end of this notebook you will be able to:</b>
<ol>
<li>Inspect an image dataset qualitatively and quantitatively, and say whether it fits a task.</li>
<li>Find and fix common dataset errors (duplicates, flipped/rotated copies, broken files, bad labels or
masks) before they leak into training.</li>
<li>Choose a split and a metric that match the use case, and explain how imbalance, object size and leakage
look different for classification, detection and segmentation.</li>
</ol>
Questions are marked 🟢 beginner · 🟡 intermediate · 🔴 advanced (optional).
</font></td></tr></table>

**Before you start:** you need a Google account (this notebook runs in Colab, a CPU runtime is enough),
Python basics, and the lecture on image datasets. No model is trained here; that is next week.
The notebook follows the first part of Karpathy's
[*A Recipe for Training Neural Networks*](https://karpathy.github.io/2019/04/25/recipe/):
know your data before you touch a model.

## 0. Setup & the park

**The Wildwood Farm Park** has cameras in four places, and it wants three things from them.

<table width="100%"><tr>
<td align="center" width="25%"><font size="6">🐑🐴</font><br><b>Farm paddock</b><br>sheep, horses</td>
<td align="center" width="25%"><font size="6">🐦</font><br><b>Aviary pond</b><br>birds</td>
<td align="center" width="25%"><font size="6">🦓</font><br><b>Savanna enclosure</b><br>zebras</td>
<td align="center" width="25%"><font size="6">🐱🐶</font><br><b>Visitor yard</b><br>cats and dogs wander
through</td>
</tr></table>

1. **Recognise** (classification): "who is in front of this camera?" The case that matters is **a dog in the
   sheep paddock**, which triggers an alert. Missing that dog is worse than a false alarm.
2. **Count** (detection): a daily headcount of sheep, zebras and birds. Far-away birds are tiny.
3. **Outline** (segmentation): how much of each enclosure each animal uses. Birds cover few pixels.

The park has no training data yet, so we practise on a **Play Dataset**: COCO 2017 photos of the same six
animals (cat, dog, bird, horse, sheep, zebra), each photo showing one animal class. Everything you learn
here is a checklist you will reuse on your own project dataset in the last section.

**Run the next cell once.** It installs the one package Colab lacks, fixes the random seed, sets the
notebook's colours and defines `use()`, the helper that lets you write small functions without ever
blocking "Run all".

In [ ]:
# @title Setup (run me first) { display-mode: "form" }
import ast
import inspect
import os
import random
import sys
import textwrap

import matplotlib as mpl
import matplotlib.pyplot as plt  # noqa: F401  (every later section plots with plt)
import numpy as np

IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("NB_SMOKE") == "1"  # automated tests: tiny data, CPU, seconds. Never set this by hand.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

if IN_COLAB:  # Jupytext turns the commented magic below into a real `%pip install -q imagehash` (N-021)
    pass
    %pip install -q imagehash

# --- Theme (SPEC L-003): white background, dark teal text, light grid, one colour per class everywhere ---
CLASSES = ["cat", "dog", "bird", "horse", "sheep", "zebra"]
CLASS_COLORS = {
    "cat": "#E07A5F",
    "dog": "#3D405B",
    "bird": "#F2CC8F",
    "horse": "#81B29A",
    "sheep": "#9C89B8",
    "zebra": "#2A9D8F",
}
INK = "#21434A"
mpl.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "text.color": INK,
        "axes.labelcolor": INK,
        "axes.edgecolor": INK,
        "axes.titlecolor": INK,
        "xtick.color": INK,
        "ytick.color": INK,
        "axes.grid": True,
        "grid.color": "#DDE5E7",
        "grid.linewidth": 0.8,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.prop_cycle": mpl.cycler(color=[CLASS_COLORS[c] for c in CLASSES]),
        "figure.dpi": 100,
        "font.size": 11,
    }
)

# --- Results every section writes into (SPEC §9) ---
RESULTS = {}
RESULTS["class_colors"] = dict(CLASS_COLORS)


# --- use(): run a learner's function next to the reference solution (N-029) ---
def _is_stub(fn) -> bool:
    """True if fn's body is only a docstring plus `...` or `pass` (the learner has not written it yet)."""
    try:
        node = ast.parse(textwrap.dedent(inspect.getsource(fn))).body[0]
        body = node.body
    except (OSError, TypeError, SyntaxError, IndexError, AttributeError):
        return False
    if body and isinstance(body[0], ast.Expr) and isinstance(getattr(body[0].value, "value", None), str):
        body = body[1:]  # skip the docstring
    return all(
        isinstance(n, ast.Pass) or (isinstance(n, ast.Expr) and getattr(n.value, "value", None) is Ellipsis)
        for n in body
    )


def _same(a, b) -> bool:
    if hasattr(a, "equals"):  # pandas
        return bool(a.equals(b))
    try:
        if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
            return bool(np.array_equal(a, b))
        return bool(a == b)
    except Exception:
        return False


def _short(x, n=60) -> str:
    s = repr(x)
    return s if len(s) <= n else s[: n - 1] + "…"


def _diff(mine, ref) -> str:
    """One short line on how `mine` differs from `ref` (e.g. 'cat: 12 vs 14')."""
    if isinstance(mine, dict) and isinstance(ref, dict):
        keys = list(ref) + [k for k in mine if k not in ref]
        bad = [k for k in keys if not _same(mine.get(k, "—"), ref.get(k, "—"))]
        return ", ".join(f"{k}: {_short(mine.get(k, '—'))} vs {_short(ref.get(k, '—'))}" for k in bad[:4])
    if isinstance(mine, (list, tuple, set)) and isinstance(ref, (list, tuple, set)):
        try:
            extra, missing = set(mine) - set(ref), set(ref) - set(mine)
            parts = []
            if missing:
                parts.append(f"missing {_short(sorted(missing, key=str))}")
            if extra:
                parts.append(f"extra {_short(sorted(extra, key=str))}")
            return " · ".join(parts) or f"same items, different order or length ({len(mine)} vs {len(ref)})"
        except TypeError:
            pass
    return f"{_short(mine)} vs {_short(ref)}"


def use(fn, *args, reference, **kwargs):
    """Run `fn(*args, **kwargs)` next to `reference(*args, **kwargs)`; ALWAYS return the reference's result.

    - stub (body is `...`) or NotImplementedError → "using the reference solution";
    - otherwise ✅ when both agree, ❌ with a short diff when they do not.
    The notebook continues with the reference either way, so "Run all" never breaks on your code.
    """
    expected = reference(*args, **kwargs)
    name = getattr(fn, "__name__", "your function")
    if _is_stub(fn):
        print(f"✍️  {name}: not written yet, using the reference solution.")
        return expected
    try:
        mine = fn(*args, **kwargs)
    except NotImplementedError:
        print(f"✍️  {name}: not written yet, using the reference solution.")
        return expected
    except Exception as err:  # a learner's bug must not stop the notebook
        print(f"❌ {name} raised {type(err).__name__}: {err} · using the reference solution.")
        return expected
    if _same(mine, expected):
        print(f"✅ {name} matches the reference solution.")
    else:
        print(f"❌ {name} differs from the reference: {_diff(mine, expected)} · using the reference.")
    return expected


print(f"Colab: {IN_COLAB} · smoke mode: {SMOKE} · seed: {SEED} · imagehash: ", end="")
try:
    import imagehash  # noqa: F401

    print("ok")
except ImportError:
    print("missing (run this cell again)")

### How the ✍️ cells work

About four times in this notebook you write a small function. Each one comes with its signature and a
`use(my_function, …, reference=…)` line that compares your result with the reference solution and
then continues with the reference, so an unfinished or wrong function never stops the notebook.
Run the cell below as is, then fill in the body and run it again to see the message change.

In [ ]:
def double(x):
    """Return twice x."""
    ...


def _ref_double(x):
    return 2 * x


result = use(double, 21, reference=_ref_double)
print("the notebook continues with:", result)

### The Play Dataset

The next cell fetches the annotation subset we prepared (`data/coco_animals_subset.json`, COCO 2017 val,
six animal classes, each image shows exactly one of them) and downloads the photos themselves from
`images.cocodataset.org`. **The first run takes a minute or two** (about 115 MB); afterwards the images
are cached and the cell is instant. The result is two tidy tables, `images` (one row per photo) and
`annotations` (one row per animal: its box and mask).

In [ ]:
# @title Data helpers (download with cache, build the tables) { display-mode: "form" }
import json
import urllib.error
import urllib.request
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd

# Where the committed manifests live. In Colab they are fetched from the repository; locally they are read
# from ./data. ⬇️ DATA_BASE_URL: change this if the repository moves or you work on a fork.
DATA_BASE_URL = "https://raw.githubusercontent.com/FHNW-VISED/eda-teaching-notebook/main/data/"
DATA_DIR = Path("data")
IMAGE_CACHE = DATA_DIR / "cache" / "coco"
SMOKE_IMAGES_PER_CLASS = 6


def _github_headers() -> dict:
    """Authorization header from the Colab secret GITHUB_TOKEN, if the repository is private (SPEC §5)."""
    if not IN_COLAB:
        return {}
    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN")
        return {"Authorization": f"token {token}"} if token else {}
    except Exception:
        return {}


def download_file(url: str, dest: Path, headers: dict | None = None, attempts: int = 3) -> Path | None:
    """Download `url` to `dest` once; a half-finished download is written to `.part` first (N-019)."""
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_suffix(dest.suffix + ".part")
    for attempt in range(attempts):
        try:
            req = urllib.request.Request(url, headers=headers or {})
            with urllib.request.urlopen(req, timeout=60) as resp, open(part, "wb") as fh:
                fh.write(resp.read())
            part.rename(dest)
            return dest
        except (urllib.error.URLError, TimeoutError, OSError):
            if attempt == attempts - 1:
                part.unlink(missing_ok=True)
                return None
    return None


def fetch_manifest(name: str) -> Path:
    """Return the local path of a committed manifest, downloading it from DATA_BASE_URL if needed."""
    path = DATA_DIR / name
    if not path.exists():
        if download_file(DATA_BASE_URL + name, path, headers=_github_headers()) is None:
            raise RuntimeError(
                f"Could not download {name} from {DATA_BASE_URL}. Check DATA_BASE_URL, or add a GITHUB_TOKEN "
                "secret in Colab if the repository is private."
            )
    return path


def download_images(rows: list[dict], workers: int = 16) -> dict[int, Path]:
    """Download every image in `rows` ('id', 'coco_url', 'file_name') into IMAGE_CACHE; one progress line."""
    done, paths = 0, {}

    def one(row):
        return row["id"], download_file(row["coco_url"], IMAGE_CACHE / row["file_name"])

    with ThreadPoolExecutor(max_workers=workers) as pool:
        for image_id, path in pool.map(one, rows):
            done += 1
            if path is not None:
                paths[image_id] = path
            print(f"\r  images ready: {done}/{len(rows)}", end="", flush=True)
    print()
    return paths


def build_tables(coco: dict, paths: dict[int, Path]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Tidy tables from a COCO-format dict: `images` (one row per photo), `annotations` (one per animal)."""
    cat_name = {c["id"]: c["name"] for c in coco["categories"]}
    lic_name = {lic["id"]: lic["name"] for lic in coco.get("licenses", [])}
    anns = pd.DataFrame(
        {
            "id": a["id"],
            "image_id": a["image_id"],
            "class": cat_name[a["category_id"]],
            "bbox": a["bbox"],
            "area": a["area"],
            "segmentation": a["segmentation"],
            "iscrowd": a["iscrowd"],
        }
        for a in coco["annotations"]
        if a["image_id"] in paths
    )
    per_image_class = anns.groupby("image_id")["class"].first()
    n_animals = anns.groupby("image_id").size()
    imgs = pd.DataFrame(
        {
            "id": im["id"],
            "file": im["file_name"],
            "path": str(paths[im["id"]]),
            "width": im["width"],
            "height": im["height"],
            "class": per_image_class.get(im["id"]),
            "n_animals": int(n_animals.get(im["id"], 0)),
            "license": lic_name.get(im["license"], str(im["license"])),
        }
        for im in coco["images"]
        if im["id"] in paths
    )
    imgs = imgs.sort_values("id").reset_index(drop=True)
    anns = anns.sort_values("id").reset_index(drop=True)
    return imgs, anns

In [ ]:
# Load the manifest, pick the images (all of them, or 6 per class in smoke mode), download, build the tables.
COCO_RAW = json.loads(fetch_manifest("coco_animals_subset.json").read_text())
_cat_name = {c["id"]: c["name"] for c in COCO_RAW["categories"]}
_image_class = {a["image_id"]: _cat_name[a["category_id"]] for a in COCO_RAW["annotations"]}

_rows = COCO_RAW["images"]
if SMOKE:  # same code path, fewer images (N-014)
    _rng = random.Random(SEED)
    _rows = []
    for c in CLASSES:
        _of_class = [im for im in COCO_RAW["images"] if _image_class[im["id"]] == c]
        _rows += sorted(_rng.sample(_of_class, SMOKE_IMAGES_PER_CLASS), key=lambda im: im["id"])

print(f"{len(_rows)} images to fetch from images.cocodataset.org (cached in {IMAGE_CACHE}/)")
_paths = download_images(_rows)
if len(_paths) < len(_rows):
    print(f"  ⚠️ {len(_rows) - len(_paths)} images could not be downloaded and are left out; run again.")

images, annotations = build_tables(COCO_RAW, _paths)
RESULTS["n_images"] = int(len(images))

_counts = Counter(images["class"])
print(f"images: {len(images)} · animals (annotations): {len(annotations)}")
print("per class: " + " · ".join(f"{c} {_counts[c]}" for c in CLASSES))
images.head()

Each row of `images` is one photo with its size, its class and how many animals it holds; `annotations`
has one row per animal with its box and mask. Look at the `n_animals` column: several photos hold more
than one animal, which already hints at a question we will meet in section 4.

## 1. Image datasets

**Objective:** know where image datasets come from and what makes a good one, then hold the Play Dataset
against that checklist.

The park has no training data yet. Before anyone mounts cameras and hires annotators, the first question is
always: *does a usable dataset already exist?* These are the places to look, and the tools to label your own
images when nothing fits.

**Find a dataset:**
[Kaggle datasets](https://www.kaggle.com/datasets) ·
[Google Dataset Search](https://datasetsearch.research.google.com) ·
[V7 open datasets](https://www.v7labs.com/open-datasets) ·
[Roboflow Universe](https://universe.roboflow.com) ·
[LILA](https://lila.science) (wildlife and camera-trap data) ·
[V7 blog on computer-vision datasets](https://www.v7labs.com/blog) ·
[torchvision datasets](https://pytorch.org/vision/stable/datasets.html) (the classics, one import away).

**Label your own images:**
[Supervisely](https://supervisely.com) ·
[V7](https://www.v7labs.com) (free for education) ·
[Roboflow Annotate](https://roboflow.com/annotate) ·
[Label Studio](https://labelstud.io) (open source).

Our Play Dataset is a slice of [COCO](https://cocodataset.org), and the woodland trail-camera images of
section 6 come from LILA. Part of "finding" a dataset is reading its licence: COCO's annotations are
CC BY 4.0, but every photo keeps its own Flickr licence, which is why this notebook downloads the images at
runtime instead of shipping them.

Having a dataset is not the same as having a *good* one. This is the checklist from the lecture:

<table width="100%"><tr><td bgcolor="#E3F2FD"><font color="#0D47A1">
<b>📘 Theory · What makes a good dataset</b><br>
<ul>
<li><b>Does it fit the question?</b> The images, the classes and the setting must match the job you want the
model to do.</li>
<li><b>Enough samples:</b> a few hundred to a few thousand images, depending on how many classes you
have.</li>
<li><b>2–10 classes.</b></li>
<li><b>About 500×500 px.</b> Check the input size of the models you plan to use.</li>
<li><b>Classes, distributions, imbalance, bias.</b> Know them, and take them into account in training
(for example with a weighted loss).</li>
<li><b>Variability matches the target population.</b> Lighting, viewpoints, backgrounds and seasons in the
data should look like the ones the model will meet.</li>
<li><b>Labels are present and validated.</b></li>
<li><b>Objects are identifiable and separable by eye.</b> If you cannot tell them apart, the model will
struggle too.</li>
<li><b>What do experts and the literature say</b> about this data and this task?</li>
<li><b>Get a qualitative and a quantitative sense of the data</b> (Karpathy, 2019): look at many images,
then measure.</li>
</ul>
</font></td></tr></table>

The next cell goes through the checklist for the Play Dataset. Where the tables from section 0 can answer
a bullet it computes the answer (✅ fine, ⚠️ needs attention); where only your eyes can, it says so (👀)
and points to the section that will look.

<table width="100%"><tr><td bgcolor="#E0F7F4"><font color="#00564D">
<b>❓ Questions</b><br>
Predict before you run: the park wants to recognise, count and outline six animals from fixed cameras.
</font></td></tr></table>

**Q1.** 🟢 Which bullets do you expect the Play Dataset to pass (✅) as it is, and which one worries you
most for the dog-in-the-paddock alert? Write one line, then run the cell.

In [ ]:
# Q1  Apply the checklist to the Play Dataset. Every number comes from the `images` table of section 0.
n_images = len(images)
counts = images["class"].value_counts().reindex(CLASSES).fillna(0).astype(int)
n_classes = int((counts > 0).sum())
imbalance = counts.max() / max(counts.min(), 1)
median_w, median_h = int(images["width"].median()), int(images["height"].median())
share_labelled = float((images["n_animals"] > 0).mean())
share_several = float((images["n_animals"] > 1).mean())

checklist = [
    (
        "⚠️",
        "Fits the question?",
        "the park's six animals, but Flickr holiday photos, not fixed cameras → judge in section 2",
    ),
    (
        "✅" if n_images >= 300 else "⚠️",
        "A few hundred to a few thousand samples",
        f"{n_images} images" + ("" if n_images >= 300 else " (a tiny test sample; the full set has ~600)"),
    ),
    ("✅" if 2 <= n_classes <= 10 else "⚠️", "2–10 classes", f"{n_classes}: " + ", ".join(CLASSES)),
    (
        "✅" if min(median_w, median_h) >= 224 else "⚠️",
        "About 500×500 px",
        f"median {median_w}×{median_h} px; most classifiers take 224–512 px, so there is room to resize",
    ),
    (
        "⚠️" if imbalance > 2 else "✅",
        "Classes, distributions, imbalance, bias",
        " · ".join(f"{c} {counts[c]}" for c in CLASSES)
        + f" → largest / smallest = {imbalance:.1f}× (weighted loss? section 4)",
    ),
    ("👀", "Variability matches the target population", "lighting, distance, backgrounds: look in section 2"),
    (
        "✅" if share_labelled == 1 else "⚠️",
        "Labels present",
        f"{share_labelled:.0%} of images have at least one box and mask",
    ),
    (
        "⚠️",
        "Labels validated",
        "COCO-ReM (Singh et al., 2024) found ~2.4 % of val masks poor, plus duplicate masks → sections 5, 9",
    ),
    (
        "👀",
        "Objects identifiable and separable by eye",
        f"{share_several:.0%} of images hold several animals; check the grids in section 2",
    ),
    (
        "📚",
        "What experts and the literature say",
        "Lin et al. 2014 (COCO) and Singh et al. 2024 (COCO-ReM) document the data and its mask errors",
    ),
    ("📓", "Qualitative and quantitative sense (Karpathy 2019)", "sections 2–4 of this notebook"),
]
for status, bullet, finding in checklist:
    print(f"{status} {bullet}: {finding}")

Read the list top to bottom: the size, class count and resolution pass on their own, and everything that
needs attention points to a later section of this notebook, which is the order we will follow.

**Worked answer (Q1).** Sample size, number of classes and resolution pass without any work. The bullets
that matter for the alert are *fit* and *imbalance*: holiday photos of dogs are not a camera pointed at a
paddock, and on the full set the biggest class has more than twice the images of the smallest, so a model
can score well while ignoring a rare animal. Section 4 makes that concrete for the dog and section 7 picks
the metric for it.

## 2. Become one with the data

**Objective:** look at many images before computing anything, and write down what you see.

Section 1 ended with three 👀 bullets that no table can answer: does the data *fit* the park's cameras, does
its *variability* match what the cameras will see, and are the animals *identifiable by eye*? The only tool
for those is your eyes, and this is the step most people skip. Karpathy puts it first in his recipe:

<table width="100%"><tr><td bgcolor="#E8F5E9"><font color="#1B5E20">
<b>🧾 Recipe · Step 1: Become one with the data</b><br>
<i>"The first step to training a neural net is to not touch any neural net code at all and instead begin by
thoroughly inspecting your data. This step is critical. I like to spend copious amount of time (measured in
units of hours) scanning through thousands of examples, understanding their distribution and looking for
patterns."</i><br>
Andrej Karpathy, <a href="https://karpathy.github.io/2019/04/25/recipe/">A Recipe for Training Neural
Networks</a> (2019).
</font></td></tr></table>

We will not spend hours, but we will spend a few minutes. The first cell draws one row of photos per class,
each framed in its class colour (the same colour every plot in this notebook uses for that class). Scroll
slowly: look at the backgrounds, the distance to the animal, the lighting, and whether anything else is in
the photo.

In [ ]:
# @title Grid helpers { display-mode: "form" }
from IPython.display import display
from PIL import Image


def load_thumb(path, size: int = 256) -> Image.Image:
    """Open an image as a small RGB thumbnail (fast: JPEG decodes at reduced size)."""
    im = Image.open(path)
    im.draft("RGB", (size, size))
    im = im.convert("RGB")
    im.thumbnail((size, size))
    return im


def sample_rows(class_name: str, n: int) -> pd.DataFrame:
    """A fixed random sample of `n` rows of `images` for one class (seeded: re-runs show the same photos)."""
    of_class = images[images["class"] == class_name]
    n = min(n, len(of_class))
    idx = random.Random(SEED).sample(sorted(of_class.index), n)
    return of_class.loc[sorted(idx)]


def frame(ax, color: str, width: float = 4) -> None:
    """Draw a coloured frame around an image axis and hide ticks and grid."""
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor(color)
        spine.set_linewidth(width)


def show_grid(class_name: str, n: int, cols: int = 6, label_animals: bool = True) -> None:
    """Show `n` photos of one class in a grid, framed in CLASS_COLORS[class_name]."""
    rows_df = sample_rows(class_name, n)
    n = len(rows_df)
    n_rows = max(1, -(-n // cols))
    fig, axes = plt.subplots(n_rows, cols, figsize=(2.2 * cols, 2.4 * n_rows))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis("off")
    for ax, (_, row) in zip(axes, rows_df.iterrows(), strict=False):
        ax.axis("on")
        ax.imshow(load_thumb(row["path"]))
        frame(ax, CLASS_COLORS[class_name])
        if label_animals:
            plural = "s" if row["n_animals"] != 1 else ""
            ax.set_title(f"{row['n_animals']} {class_name}{plural}", fontsize=8)
    fig.suptitle(f"{class_name} · {n} of {int((images['class'] == class_name).sum())} photos", color=INK)
    fig.tight_layout()
    display(fig)
    plt.close(fig)


def show_all_classes(n_per_class: int) -> None:
    """One row per class, `n_per_class` photos each, every row framed in its class colour."""
    fig, axes = plt.subplots(len(CLASSES), n_per_class, figsize=(2.0 * n_per_class, 2.1 * len(CLASSES)))
    axes = np.atleast_2d(axes)
    for r, class_name in enumerate(CLASSES):
        rows_df = sample_rows(class_name, n_per_class)
        for c in range(n_per_class):
            ax = axes[r, c]
            ax.axis("off")
            if c < len(rows_df):
                ax.axis("on")
                ax.imshow(load_thumb(rows_df.iloc[c]["path"]))
                frame(ax, CLASS_COLORS[class_name])
            if c == 0:
                ax.set_ylabel(class_name, color=CLASS_COLORS[class_name], fontsize=12, fontweight="bold")
    fig.tight_layout()
    display(fig)
    plt.close(fig)

<table width="100%"><tr><td bgcolor="#E0F7F4"><font color="#00564D">
<b>❓ Questions</b><br>
Predict before you run: the park's cameras are fixed, outdoors, and often far from the animal.
</font></td></tr></table>

**Q2.** 🟢 Which class do you expect to look *least* like a fixed park camera's view, and why? One line,
then run the cell.

In [ ]:
# Q2  One row per class: a fixed sample of photos, framed in the class colour used throughout the notebook.
N_PER_CLASS = 3 if SMOKE else 6  # fewer thumbnails in smoke mode, same code (N-014)
show_all_classes(N_PER_CLASS)

💡 Even six photos per class show that these are holiday snapshots, not camera-trap frames: close-ups,
people, sofas and streets, with the animal usually centred and filling the frame.

**Worked answer (Q2).** Cats and dogs are the farthest from a park camera: they are photographed indoors,
up close, often with people and furniture. Sheep, horses and zebras are at least outdoors and sometimes far
away, which is closer to what the paddock and savanna cameras will see.

Now pick one class and look at more of it. Twelve photos is enough to start noticing patterns; raise it if
you have time.

<table width="100%"><tr><td bgcolor="#FFF4D6"><font color="#7A4A00">
<b>✏️ Your turn · Pick a class</b><br>
Set <code>CLASS_TO_SHOW</code> to one of <code>cat, dog, bird, horse, sheep, zebra</code> and
<code>N_SHOW</code> to how many photos you want (up to the class size), then run the cell. Try at least
<code>"dog"</code> (the alert class) and <code>"bird"</code> (the tiny-box class).
</font></td></tr></table>

In [ ]:
# ✏️ YOUR TURN
CLASS_TO_SHOW = "dog"
N_SHOW = 12

if CLASS_TO_SHOW not in CLASSES:
    raise SystemExit(f"CLASS_TO_SHOW must be one of {CLASSES}, not {CLASS_TO_SHOW!r}")
if not isinstance(N_SHOW, int) or N_SHOW < 1:
    raise SystemExit(f"N_SHOW must be a positive integer, not {N_SHOW!r}")
n_show = min(N_SHOW, 6) if SMOKE else N_SHOW  # smoke mode caps the grid, same code path (N-014)
show_grid(CLASS_TO_SHOW, n_show)

💡 The title of each photo says how many animals of that class it holds: more than one is common, so "one
label per image" is already a simplification, and section 4 asks which label a photo with several animals
should get.

<table width="100%"><tr><td bgcolor="#E0F7F4"><font color="#00564D">
<b>❓ Questions</b><br>
Karpathy's step ends with writing things down. Do the same: edit the next markdown cell (double-click it).
</font></td></tr></table>

**Q3.** 🟢 Write three observations about the photos you just looked at. Good observations are specific
("many dog photos are indoors with a person's legs in the frame"), and each one hints at something the park
should worry about or check later (fit to the cameras, lighting, distance, several animals, people, odd
photos).

**My three observations** (double-click to edit):

1. …
2. …
3. …

**Worked answer (Q3)**, for the dog grid on the full set. Yours will differ; that is the point.

1. **Setting:** most dog photos are indoors or in a street, taken by the owner from a metre away. A paddock
   camera sees a small dog far away in grass. The model will have to generalise from close-ups to that
   (section 11's question "does it fit?").
2. **People:** a person, a hand or a leash appears in many photos. A model can learn "person + leash = dog"
   and miss the stray dog alone in the paddock.
3. **Several animals and odd photos:** some photos hold two or three dogs, some show a dog in a costume,
   on a skateboard or on a screen. Each of these is a labelling decision (sections 4 and 5) before it is a
   modelling problem.

Keep your three lines: section 11 asks you for the same three on your own dataset, and the rest of this
notebook turns the observations above into numbers, one at a time.

## 3. Image inspection

**Objective:** measure sizes, aspect ratios, colour modes and colour statistics, and learn to read a cluster
of odd values as a clue about where the photos came from.

Your eyes found patterns in section 2. Now the computer looks at every photo once and writes down a handful
of numbers per image: its size, its shape, how its colours are stored and how bright it is. None of these
numbers needs a label, so this is the first check you can run on *any* folder of images, including your own
project dataset in section 11. Karpathy's recipe says why this pays off:

<table width="100%"><tr><td bgcolor="#E8F5E9"><font color="#1B5E20">
<b>🧾 Recipe · Step 1, continued: search, filter, sort</b><br>
<i>"Once you get a qualitative sense it is also a good idea to write some simple code to search/filter/sort
by whatever you can think of (e.g. type of label, size of annotations, number of annotations, etc.) and
visualize their distributions and the outliers along any axis. The outliers especially almost always uncover
some bugs in data quality or preprocessing."</i><br> Andrej Karpathy, <a
href="https://karpathy.github.io/2019/04/25/recipe/">A Recipe for Training Neural Networks</a> (2019).
</font></td></tr></table>

The next cell (collapsed) defines one function that measures a single photo. The cell after it runs that
function on every photo in the Play Dataset and collects the numbers in a table called `image_stats`.

In [ ]:
# @title Measurement helpers { display-mode: "form" }
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from matplotlib.patches import Rectangle
from PIL import ImageDraw

HIST_BINS = 32
CHANNEL_COLORS = {"R": "#C0392B", "G": "#27AE60", "B": "#2980B9"}
COLOUR_MODES = ["RGB", "L", "RGBA", "CMYK"]  # the modes we always report, even when the count is 0


def measure_image(path, thumb: int) -> tuple[dict, np.ndarray]:
    """Open one photo and return (stats, channel_hist).

    stats: width, height, mode (as stored in the file), aspect (width / height), long_side, megapixels,
    brightness (mean grey level 0–255 on a thumbnail), contrast (its standard deviation), colourfulness
    (mean distance of the RGB channels from grey: 0 for a grayscale picture), mean_r / mean_g / mean_b.
    channel_hist: a (3, HIST_BINS) array, the density histogram of the R, G and B channels.
    """
    with Image.open(path) as im:
        mode, (w, h) = im.mode, im.size  # read these before decoding: they describe the file itself
        im.draft("RGB", (thumb, thumb))  # fast reduced-size JPEG decode (no effect on other formats)
        rgb = im.convert("RGB")
        rgb.thumbnail((thumb, thumb))
        arr = np.asarray(rgb, dtype=np.float32)
    grey = arr.mean(axis=2)
    channels = arr.reshape(-1, 3)
    hist = np.stack(
        [np.histogram(channels[:, k], bins=HIST_BINS, range=(0, 256), density=True)[0] for k in range(3)]
    )
    stats = {
        "width": w,
        "height": h,
        "mode": mode,
        "aspect": w / h,
        "long_side": max(w, h),
        "megapixels": w * h / 1e6,
        "brightness": float(grey.mean()),
        "contrast": float(grey.std()),
        "colourfulness": float(np.abs(channels - grey.reshape(-1, 1)).mean()),
        "mean_r": float(channels[:, 0].mean()),
        "mean_g": float(channels[:, 1].mean()),
        "mean_b": float(channels[:, 2].mean()),
    }
    return stats, hist


def show_rows(rows_df: pd.DataFrame, title: str, caption=None, cols: int = 6) -> None:
    """A small gallery of the photos in `rows_df` (rows of image_stats), each framed in its class colour.

    `caption(row) -> str` writes the title above each photo; by default the size and the colour mode.
    """
    if caption is None:

        def caption(row):
            return f"{row['class']} · {row['width']}×{row['height']} {row['mode']}"

    n = len(rows_df)
    if n == 0:
        print(f"{title}: no photos to show.")
        return
    cols = min(cols, n)
    n_rows = -(-n // cols)
    fig, axes = plt.subplots(n_rows, cols, figsize=(2.4 * cols, 2.7 * n_rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, (_, row) in zip(axes.ravel(), rows_df.iterrows(), strict=False):
        ax.axis("on")
        ax.imshow(load_thumb(row["path"]))
        frame(ax, CLASS_COLORS[row["class"]])
        ax.set_title(caption(row), fontsize=8)
    fig.suptitle(title, color=INK)
    fig.tight_layout()
    display(fig)
    plt.close(fig)


def show_extremes(stat: str, n: int = 4) -> None:
    """Gallery of the `n` photos with the lowest and the `n` with the highest value of `stat`.

    `stat` is a numeric column of `image_stats` (e.g. "brightness", "aspect", "megapixels", "colourfulness").
    """
    numeric = [c for c in image_stats.columns if pd.api.types.is_numeric_dtype(image_stats[c]) and c != "id"]
    if stat not in numeric:
        raise SystemExit(f"stat must be one of {numeric}, not {stat!r}")
    ordered = image_stats.sort_values(stat)
    n = max(1, min(n, len(ordered) // 2))

    def caption(row):
        return f"{row['class']} · {stat} = {row[stat]:.2f}\n{row['width']}×{row['height']} {row['mode']}"

    show_rows(ordered.head(n), f"lowest {stat}", caption, cols=n)
    show_rows(ordered.tail(n)[::-1], f"highest {stat}", caption, cols=n)

In [ ]:
# Measure every photo once (a few seconds: each JPEG is decoded at thumbnail size) and keep the numbers.
THUMB = 64 if SMOKE else 128  # smaller thumbnails in smoke mode, same code (N-014)

_records, _hists = [], []
for _, _row in images.iterrows():
    _stats, _hist = measure_image(_row["path"], THUMB)
    _records.append({"id": _row["id"], "class": _row["class"], "path": _row["path"], **_stats})
    _hists.append(_hist)
image_stats = pd.DataFrame(_records)
CHANNEL_HISTS = np.stack(_hists)  # (n_images, 3, HIST_BINS)
RESULTS["image_stats_n"] = int(len(image_stats))

_same_size = image_stats[["width", "height"]].to_numpy() == images[["width", "height"]].to_numpy()
_agree = int(_same_size.all(axis=1).sum())
print(f"measured {len(image_stats)} photos · the manifest's width/height matches the file for {_agree}")
STAT_COLUMNS = ["width", "height", "aspect", "megapixels", "brightness", "contrast", "colourfulness"]
image_stats[STAT_COLUMNS].describe().round(2)

💡 `describe()` is the quickest sanity check there is: look at `min` and `max` of every column before you
look at anything else. A `min` width of 0, a `max` of 6000 or a brightness of 254 each tell a story.

### Sizes and shapes

A model sees every photo at one fixed size, so the first question is how far the photos are from that.

<table width="100%"><tr><td bgcolor="#E0F7F4"><font color="#00564D">
<b>❓ Questions</b><br>
Predict before you run. These photos were uploaded to Flickr by hundreds of people and collected by the COCO
team; the park's photos will come from a few fixed cameras.
</font></td></tr></table>

**Q4.** 🟢 Do you expect all photos to have the same size? Sketch (in your head) where the dots of a
width-vs-height scatter would land: one point, a line, a cloud? Then run the cell.

In [ ]:
# Q4  Left: every photo as a dot (width across, height up) in its class colour; the dashed line is a square
#     photo. Right: the aspect ratio width/height, stacked by class.
fig, (ax_sc, ax_hist) = plt.subplots(1, 2, figsize=(13, 4.8))
for c in CLASSES:
    s = image_stats[image_stats["class"] == c]
    ax_sc.scatter(s["width"], s["height"], s=22, alpha=0.65, color=CLASS_COLORS[c], label=c, edgecolor="none")
_lim = float(max(image_stats["width"].max(), image_stats["height"].max())) * 1.08
ax_sc.plot([0, _lim], [0, _lim], "--", color=INK, lw=1)
ax_sc.set_xlim(0, _lim)
ax_sc.set_ylim(0, _lim)
ax_sc.set_xlabel("width (px)")
ax_sc.set_ylabel("height (px)")
ax_sc.set_title("Image sizes")
ax_sc.legend(frameon=False, fontsize=9, markerscale=1.6)

_bins = np.linspace(image_stats["aspect"].min() * 0.98, image_stats["aspect"].max() * 1.02, 28)
ax_hist.hist(
    [image_stats.loc[image_stats["class"] == c, "aspect"] for c in CLASSES],
    bins=_bins,
    stacked=True,
    color=[CLASS_COLORS[c] for c in CLASSES],
    label=CLASSES,
)
for ratio, name in [(3 / 4, "3:4"), (1.0, "1:1"), (4 / 3, "4:3")]:
    ax_hist.axvline(ratio, color=INK, lw=0.8, ls=":")
    ax_hist.text(ratio, ax_hist.get_ylim()[1] * 0.97, name, ha="center", va="top", fontsize=8, color=INK)
ax_hist.set_xlabel("aspect ratio = width / height  (left of 1: portrait, right: landscape)")
ax_hist.set_ylabel("photos")
ax_hist.set_title("Aspect ratios")
fig.tight_layout()
display(fig)
plt.close(fig)

💡 The dots do not fill the plane: they sit on a few lines and clusters, because the photos share a common
longest side. The aspect histogram shows the same thing from the side: a landscape peak, a portrait peak and
a long tail of panoramas and tall crops. Nothing here is "wrong", but it means a model fed 224×224 squares
will see most of these photos squashed or cropped (the 🔀 Callout at the end of this section).

**Worked answer (Q4).** Not one size, and not a random cloud either: COCO downloaded its photos from Flickr
at one standard size, so the longest side is capped at the same value for almost every photo and the dots
line up along "width = cap" and "height = cap". A dataset from the park's own cameras would look different:
one dot per camera model (every frame of a camera has the same size), which is both tidier and a leakage
risk you will meet in section 6 (the size gives away the camera).

### Colour modes and colours

How a file stores colour is easy to forget and painful to discover during training: a single grayscale (`L`)
or `RGBA` file in a batch of `RGB` arrays crashes the data loader, or silently gets a channel duplicated or
dropped. The next cell counts the modes and draws the typical RGB histogram of each class.

In [ ]:
# Colour modes per class (always listing RGB / L / RGBA / CMYK), then the mean R, G, B histogram per class.
_modes = COLOUR_MODES + sorted(set(image_stats["mode"]) - set(COLOUR_MODES))
mode_by_class = pd.crosstab(image_stats["mode"], image_stats["class"])
mode_by_class = mode_by_class.reindex(index=_modes, columns=CLASSES, fill_value=0)
image_stats["looks_gray"] = (image_stats["mode"] == "L") | (image_stats["colourfulness"] < 2.0)
print("colour modes: " + " · ".join(f"{m} {int(mode_by_class.loc[m].sum())}" for m in _modes))
_n_gray = int(image_stats["looks_gray"].sum())
print(f"photos that look grayscale (mode L, or RGB with equal channels): {_n_gray}")

fig = plt.figure(figsize=(13, 4.6))
_gs = fig.add_gridspec(
    2, 4, width_ratios=[1.4, 1, 1, 1], hspace=0.65, wspace=0.3, left=0.05, right=0.99, top=0.86, bottom=0.12
)
ax_modes = fig.add_subplot(_gs[:, 0])
_bottom = np.zeros(len(_modes))
for c in CLASSES:
    ax_modes.bar(_modes, mode_by_class[c].to_numpy(), bottom=_bottom, color=CLASS_COLORS[c], label=c)
    _bottom += mode_by_class[c].to_numpy()
ax_modes.set_ylabel("photos")
ax_modes.set_title("Colour mode as stored in the file", fontsize=10)
ax_modes.legend(frameon=False, fontsize=8)

_edges = np.linspace(0, 256, HIST_BINS + 1)
_centres = (_edges[:-1] + _edges[1:]) / 2
for k, c in enumerate(CLASSES):
    ax = fig.add_subplot(_gs[k // 3, 1 + k % 3])
    _mean_hist = CHANNEL_HISTS[(image_stats["class"] == c).to_numpy()].mean(axis=0)
    for ch_idx, ch in enumerate("RGB"):
        ax.plot(_centres, _mean_hist[ch_idx], color=CHANNEL_COLORS[ch], lw=1.4, label=ch)
    ax.set_title(f"{c} ({int((image_stats['class'] == c).sum())} photos)", color=CLASS_COLORS[c], fontsize=10)
    ax.set_yticks([])
    ax.set_xlim(0, 255)
    ax.tick_params(labelsize=7)
    for spine in ax.spines.values():
        spine.set_edgecolor(CLASS_COLORS[c])
        spine.set_linewidth(1.6)
    if k == 0:
        ax.legend(frameon=False, fontsize=7, loc="upper left")
fig.text(0.67, 0.95, "Mean R, G, B histogram per class (pixel value 0–255)", ha="center", color=INK)
display(fig)
plt.close(fig)

💡 Read the histograms as *typical colours of a class*: a class whose curves differ from the others' (a
strong blue channel for a pond, high values across all three for a sandy savanna) is a class a model can
recognise from colour alone. That is a shortcut you should know about: a dog on green grass in the paddock
may be classified as "sheep" because the model learned "green = paddock = sheep".

### Brightness and the extremes

Brightness is the mean grey level of a photo (0 black, 255 white). Its distribution tells you whether the
dataset has night shots at all, and its extremes are the fastest route to broken or odd files.

In [ ]:
# Brightness per class as box plots, in class colours. The dotted lines mark "very dark" and "very bright".
fig, ax = plt.subplots(figsize=(9, 3.8))
_data = [image_stats.loc[image_stats["class"] == c, "brightness"] for c in CLASSES]
_box = ax.boxplot(_data, tick_labels=CLASSES, patch_artist=True, widths=0.6, medianprops={"color": INK})
for patch, c in zip(_box["boxes"], CLASSES, strict=True):
    patch.set_facecolor(CLASS_COLORS[c])
    patch.set_alpha(0.85)
for level in (40, 215):
    ax.axhline(level, color=INK, lw=0.8, ls=":")
ax.set_ylim(0, 255)
ax.set_ylabel("brightness (mean grey level)")
ax.set_title("Brightness per class")
fig.tight_layout()
display(fig)
plt.close(fig)

💡 Every class covers most of the range, so the dataset has dim and bright photos, but the medians sit
comfortably in the middle: these are daytime holiday photos. The park's cameras also run at dusk and at
night, which this data cannot teach.

Numbers point, eyes decide. `show_extremes(stat, n)` shows the `n` photos at both ends of any column of
`image_stats`, so you can see *what* a very dark, very wide or very grey photo actually is.

<table width="100%"><tr><td bgcolor="#FFF4D6"><font color="#7A4A00">
<b>✏️ Your turn · Look at the extremes</b><br>
Set <code>STAT_TO_SHOW</code> to one of <code>brightness, contrast, colourfulness, aspect, megapixels,
long_side</code> and <code>N_EXTREMES</code> to how many photos per end you want, then run the cell. Try
<code>"brightness"</code> first, then <code>"colourfulness"</code> (the low end is where grayscale pictures
hide) and <code>"aspect"</code>.
</font></td></tr></table>

In [ ]:
# ✏️ YOUR TURN
STAT_TO_SHOW = "brightness"
N_EXTREMES = 4

if not isinstance(N_EXTREMES, int) or N_EXTREMES < 1:
    raise SystemExit(f"N_EXTREMES must be a positive integer, not {N_EXTREMES!r}")
n_extremes = min(N_EXTREMES, 3) if SMOKE else N_EXTREMES  # smoke mode caps the gallery, same code (N-014)
show_extremes(STAT_TO_SHOW, n_extremes)

💡 The darkest photos are usually real (an animal in shade, a night shot with flash) and worth keeping: they
are the kind of frame the park's cameras produce at dusk. The brightest are often snow, sand or an
over-exposed sky. When one end shows a uniform grey rectangle, a thumbnail-sized file or a photo of a photo,
you have found a file for section 5's cleaning list.

### The detective beat: a cluster is a clue

A statistic that most photos share, and a few do not, is more than a curiosity. Photos from one pipeline
(one camera, one download script, one phone model) come out with the same size and mode. A **cluster of odd
sizes or modes therefore points to a different source**: another camera, another download, a batch someone
converted by hand. On the park's cameras you would see it as two size clusters (two camera models) or a
group of grayscale frames (the infrared night mode). Here we look for the same thing in COCO.

<table width="100%"><tr><td bgcolor="#E0F7F4"><font color="#00564D">
<b>❓ Questions</b><br>
Look back at the scatter from Q4 before you run the next cell.
</font></td></tr></table>

**Q5.** 🟡 Which longest-side value do most photos share, and how many photos do you think do *not* share
it? Write down a guess, then run the cell: it counts them, lists their sizes and shows them.

In [ ]:
# Q5  The most common longest side, the photos that break the pattern, and the photos that look grayscale.
COMMON_LONG_SIDE = int(image_stats["long_side"].mode().iloc[0])
odd_size = image_stats[image_stats["long_side"] != COMMON_LONG_SIDE].sort_values("long_side")
gray_like = image_stats[image_stats["looks_gray"]]

print(
    f"{len(image_stats) - len(odd_size)} of {len(image_stats)} photos have a longest side of "
    f"{COMMON_LONG_SIDE} px; {len(odd_size)} do not."
)
if len(odd_size):
    _sizes = odd_size.apply(lambda r: f"{r['width']}×{r['height']}", axis=1).value_counts()
    print("odd sizes: " + " · ".join(f"{s} ×{n}" for s, n in _sizes.items()))
else:
    print("every photo in this sample went through the same resize step.")
print(f"{len(gray_like)} photos look grayscale.")

N_ODD = 6 if SMOKE else 12  # how many of the odd ones to show (N-014)
show_rows(odd_size.head(N_ODD), f"photos whose longest side is not {COMMON_LONG_SIDE} px")
show_rows(gray_like.head(N_ODD), "photos that look grayscale")

💡 The odd sizes are small, not large: nobody uploaded a 4000-pixel photo here. The photos that fall below
the common cap were *already* smaller than it when COCO fetched them, which is a hint about their origin
(old uploads, low-resolution cameras, screenshots). A grayscale photo in a colour dataset is a similar hint,
and both go on the list of things to decide about in section 5.

**Worked answer (Q5).** On the full Play Dataset almost every photo has a longest side of 640 px, because
COCO downloaded Flickr's "640 px" rendition, and only a handful fall below it (none may appear in a small
sample). The number itself matters less than the habit: whenever a column of `image_stats` has one dominant
value, ask "what made the others different?", because the answer is usually a different source, and a
different source usually means different labels, lighting or quality. On your own dataset (section 11) this
is the first question to ask about size and mode.

### Across tasks: what resizing does to boxes and masks

Everything above was about the photo. The park's second and third jobs (count, outline) attach **boxes** and
**masks** to the photo, and they are affected by the first thing every training pipeline does: resize.

<table width="100%"><tr><td bgcolor="#EDE7F6"><font color="#311B92">
<b>🔀 Across tasks · Resizing</b><br>
<ul>
<li><b>Classification:</b> squashing a 4:3 photo into a 224×224 square distorts aspect ratios. Usually
harmless for "which animal", because every photo is distorted the same way. Cropping or padding keeps the
shape at the cost of pixels or borders.</li>
<li><b>Detection:</b> box coordinates live in pixels, so a box must be rescaled with the image: x and width
by <code>new_width / width</code>, y and height by <code>new_height / height</code>. A forgotten rescale is
the classic "the boxes are all in the top-left corner" bug.</li>
<li><b>Segmentation:</b> a mask holds class ids, not colours. Resizing it with the usual smooth (bilinear)
filter invents in-between values that are no class at all; masks need <b>nearest-neighbour</b>
resizing.</li>
</ul>
</font></td></tr></table>

**Q6.** 🟡 A box is stored as `(x, y, w, h)` in pixels on a `W × H` photo, and the photo is resized to
224×224. Write the formula for the new box. Then predict: how many distinct values does the mask hold after
a nearest-neighbour resize, and after a bilinear one? Run the cell to check both.

In [ ]:
# Q6  One photo, one box and one mask, squeezed to 224×224: the box is rescaled; the mask is resized two ways.
TARGET = 224
_polygon = annotations["segmentation"].apply(lambda s: isinstance(s, list))
_ann = annotations[(annotations["iscrowd"] == 0) & _polygon].iloc[0]
_photo = images.loc[images["id"] == _ann["image_id"]].iloc[0]
_color = CLASS_COLORS[_ann["class"]]

with Image.open(_photo["path"]) as _im:
    original = _im.convert("RGB")
W, H = original.size
sx, sy = TARGET / W, TARGET / H
x, y, w, h = _ann["bbox"]
new_box = (x * sx, y * sy, w * sx, h * sy)

mask = Image.new("L", (W, H), 0)
for _poly in _ann["segmentation"]:
    ImageDraw.Draw(mask).polygon(list(zip(_poly[::2], _poly[1::2], strict=True)), fill=1)
squashed = original.resize((TARGET, TARGET), Image.BILINEAR)
mask_nearest = np.asarray(mask.resize((TARGET, TARGET), Image.NEAREST))
mask_bilinear = np.asarray(mask.point(lambda v: v * 255).resize((TARGET, TARGET), Image.BILINEAR)) / 255.0

print(f"photo {W}×{H} → {TARGET}×{TARGET}: scale x by {sx:.3f}, y by {sy:.3f}")
_old = ", ".join(f"{v:.0f}" for v in (x, y, w, h))
_new = ", ".join(f"{v:.1f}" for v in new_box)
print(f"box  (x, y, w, h) = ({_old})  →  ({_new})")
print(
    f"mask values: original {np.unique(np.asarray(mask)).tolist()} · "
    f"nearest {np.unique(mask_nearest).tolist()} · bilinear {len(np.unique(mask_bilinear))} distinct values"
)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.9))
axes[0].imshow(original)
axes[0].add_patch(Rectangle((x, y), w, h, fill=False, edgecolor=_color, lw=2))
axes[0].set_title(f"original {W}×{H} + box", fontsize=9)
axes[1].imshow(squashed)
axes[1].add_patch(Rectangle(new_box[:2], new_box[2], new_box[3], fill=False, edgecolor=_color, lw=2))
axes[1].set_title(f"resized {TARGET}×{TARGET} + rescaled box", fontsize=9)
axes[2].imshow(mask_nearest, cmap=ListedColormap(["white", _color]), vmin=0, vmax=1)
axes[2].set_title(f"mask, nearest: {len(np.unique(mask_nearest))} values", fontsize=9)
_soft_cmap = LinearSegmentedColormap.from_list("mask", ["white", _color])
axes[3].imshow(mask_bilinear, cmap=_soft_cmap, vmin=0, vmax=1)
axes[3].set_title(f"mask, bilinear: {len(np.unique(mask_bilinear))} values", fontsize=9)
for ax in axes:
    frame(ax, _color, width=2)
fig.suptitle(f"{_ann['class']}: what one resize does to the photo, its box and its mask", color=INK)
fig.tight_layout()
display(fig)
plt.close(fig)

💡 The box follows the photo as long as you scale x and y separately. The mask does not survive a smooth
resize: the bilinear version has a soft grey rim of values between 0 and 1, which are not "background" and
not "the animal". With six classes those in-between values would become ids of *other* animals.

**Worked answer (Q6).** New box = `(x · 224/W, y · 224/H, w · 224/W, h · 224/H)`; x-like values use the
horizontal factor, y-like values the vertical one, and on a landscape photo the two factors differ, so the
box is squashed exactly like the photo. Nearest-neighbour resizing keeps the two original values (0 and 1).
Bilinear resizing produces many distinct values along every edge of the animal, and the number grows with
the length of the boundary, so thin parts (legs, tails, a bird's beak) suffer most. That is why segmentation
pipelines resize images with a smooth filter and masks with nearest-neighbour.

**Takeaway.** `image_stats` costs seconds and already answers four of the checklist's questions (size,
shape, colour mode, colour) for any folder of images, labelled or not. Section 4 adds the labels and asks
the question every count depends on: how many photos of each class do we have, and does it matter?